# RD Dataset Validation — Peak vs Label vs Mask Alignment

**Purpose.** The dataset is synthetic, so we know the exact ground truth. That advantage
only holds if the simulator places target energy *where the label says it is*. A single
coding bug (transposed axes, swapped range/velocity, an off-by-one in the FFT) would make
the labels quietly wrong and corrupt every later result.

So before any CFAR / DL work we verify one thing, frame by frame:

> the brightest point of each simulated target in the Range–Doppler map sits at the
> `(range_bin, velocity_bin)` stored in its label and in its label mask.

Work through the cells top to bottom. Every step is shown as a DataFrame or a figure.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 120)

# Find the project root (the folder that contains data/), wherever the notebook runs from.
ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

DATA_ROOT = ROOT / "data"
REPORTS_DIR = ROOT / "reports" / "validation"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = [p.name for p in sorted(DATA_ROOT.iterdir()) if p.is_dir()]
print("Project root :", ROOT)
print("Reports dir  :", REPORTS_DIR)
print("Datasets     :", DATASETS)

## Step 1 — Look at a single frame

Load one frame's RD map, mask, axes and labels. `labels.csv` is the ground truth: each row
is one target with its `range_bin` / `velocity_bin`.

In [ ]:
dataset = "clean_debug"
frame = "frame_000000"
frame_dir = DATA_ROOT / dataset / "frames" / frame

rd_mag = np.load(frame_dir / "rd_map_mag.npy")   # linear magnitude (CFAR later uses this)
rd_db = np.load(frame_dir / "rd_map_db.npy")      # log magnitude (for display / DL later)
mask = np.load(frame_dir / "label_mask.npy")
range_axis = np.load(frame_dir / "range_axis.npy")
velocity_axis = np.load(frame_dir / "velocity_axis.npy")
labels = pd.read_csv(frame_dir / "labels.csv")

print("RD map shape (range x velocity):", rd_mag.shape)
print("Mask pixels set:", int(mask.sum()), "| any NaN/Inf:", not np.isfinite(rd_mag).all())
labels

## Step 2 — Visual check (the intuitive one)

Plot the RD map and draw a cyan ✕ at every label position. If the labels are correct, each
✕ lands right on top of a bright blob. This is independent of the numeric check below, so if
both agree we have strong evidence.

In [ ]:
def plot_frame(frame_dir, ax=None):
    """Plot one RD map (dB) with label markers overlaid."""
    rd_db = np.load(frame_dir / "rd_map_db.npy")
    r_ax = np.load(frame_dir / "range_axis.npy")
    v_ax = np.load(frame_dir / "velocity_axis.npy")
    labels = pd.read_csv(frame_dir / "labels.csv")

    if ax is None:
        _, ax = plt.subplots(figsize=(7, 5))
    extent = [v_ax[0], v_ax[-1], r_ax[0], r_ax[-1]]
    im = ax.imshow(rd_db, aspect="auto", origin="lower", extent=extent,
                   cmap="inferno", vmin=-60, vmax=0)
    if len(labels):
        ax.scatter(labels["velocity_mps"], labels["range_m"],
                   marker="x", s=90, linewidths=1.6, color="cyan")
    ax.set_xlabel("Velocity [m/s]")
    ax.set_ylabel("Range [m]")
    ax.set_title(f"{frame_dir.parent.parent.name} / {frame_dir.name}")
    return ax, im


ax, im = plot_frame(frame_dir)
plt.colorbar(im, ax=ax, label="Relative magnitude [dB]")
plt.show()

## Step 3 — Quantify the offset for this one frame

For each target we search a small window (±`search` bins) around its labelled cell, find the
local peak, and record the offset `(dr, dv)` in bins. We also check the mask is set there.

Key idea — **isolated vs overlapping**: the strict pass/fail only applies to *isolated*
targets. When another (usually stronger) target sits inside the window, the weaker one is
physically masked and cannot show its own peak — we mark it `overlapping/unresolved`
(informational), not failed. In the close-pair scenario the 2nd target is masked *by design*,
so it is exempted using the dataset's config.

A target **passes** if it is isolated, its peak is within `tol` bins of the label, the mask is
set, and the data is finite.

In [ ]:
def frame_alignment(frame_dir, tol_r=2, tol_v=2, search=3, neighbour_radius=5,
                    min_cell_snr_db=15.0, close_pair_mode=False):
    rd_mag = np.load(frame_dir / "rd_map_mag.npy")
    mask = np.load(frame_dir / "label_mask.npy")
    labels = pd.read_csv(frame_dir / "labels.csv")
    n_r, n_v = rd_mag.shape
    finite = bool(np.isfinite(rd_mag).all())
    noise_floor = float(np.median(rd_mag)) + 1e-12
    bins = list(zip(labels["range_bin"].astype(int), labels["velocity_bin"].astype(int)))
    snrs = labels["cell_snr_db"].tolist() if "cell_snr_db" in labels else [np.nan] * len(bins)

    if not bins:  # empty (0-target) frame: nothing to align, just check finiteness
        g_r, g_v = np.unravel_index(int(rd_mag.argmax()), rd_mag.shape)
        return pd.DataFrame([{
            "frame": frame_dir.name, "target_id": -1, "label_r": None, "label_v": None,
            "peak_r": int(g_r), "peak_v": int(g_v), "dr": None, "dv": None,
            "cell_snr_db": None, "isolated": None, "low_snr": None, "mask_ok": None,
            "finite": finite, "peak_to_floor_db": None, "passed": finite,
            "note": "empty frame (0 targets)"}])

    rows = []
    for tid, (r, v) in enumerate(bins):
        # neighbour_radius (interference) is deliberately LARGER than `search` (peak
        # window): a Blackman mainlobe reaches ~5 bins, so a neighbour just outside the
        # search window can still capture this target's peak.
        neighbour = any(
            j != tid and abs(br - r) <= neighbour_radius and abs(bv - v) <= neighbour_radius
            for j, (br, bv) in enumerate(bins)
        )
        masked_by_design = close_pair_mode and tid > 0
        isolated = not (neighbour or masked_by_design)

        # A target below ~15 dB cell SNR is not reliably visible (measured Pd < 1 there),
        # so the local peak is noise and an alignment check would be meaningless.
        snr = float(snrs[tid]) if tid < len(snrs) else np.nan
        low_snr = bool(np.isfinite(snr) and snr < min_cell_snr_db)

        r0, r1 = max(0, r - search), min(n_r, r + search + 1)
        v0, v1 = max(0, v - search), min(n_v, v + search + 1)
        patch = rd_mag[r0:r1, v0:v1]
        pr, pv = np.unravel_index(int(patch.argmax()), patch.shape)
        peak_r, peak_v = r0 + pr, v0 + pv
        dr, dv = peak_r - r, peak_v - v
        mask_ok = bool(mask[r, v])
        aligned = abs(dr) <= tol_r and abs(dv) <= tol_v

        # Strict alignment is only required for targets that are both isolated and visible.
        if isolated and not low_snr:
            passed = aligned and mask_ok and finite
            note = "" if passed else "isolated visible target misaligned"
        else:
            passed = mask_ok and finite
            note = "low SNR (informational)" if low_snr else "overlapping/unresolved (informational)"

        rows.append({
            "frame": frame_dir.name, "target_id": tid, "label_r": r, "label_v": v,
            "peak_r": peak_r, "peak_v": peak_v, "dr": dr, "dv": dv,
            "cell_snr_db": round(snr, 2) if np.isfinite(snr) else None,
            "isolated": isolated, "low_snr": low_snr, "mask_ok": mask_ok, "finite": finite,
            "peak_to_floor_db": round(20 * np.log10(rd_mag[peak_r, peak_v] / noise_floor), 1),
            "passed": passed, "note": note})
    return pd.DataFrame(rows)


frame_alignment(frame_dir)

## Step 4 — Validate a whole dataset

Run the per-frame check across every frame and stack the results into one DataFrame. We also
save it to `data/<dataset>/validation_report/alignment_report.csv` so you have a record.

In [ ]:
def dataset_is_close_pair(ddir):
    cfg = ddir / "dataset_config_used.yaml"
    return cfg.exists() and "close_pair" in cfg.read_text(encoding="utf-8")


def validate_dataset(name, tol_r=2, tol_v=2, save=True):
    ddir = DATA_ROOT / name
    close_pair = dataset_is_close_pair(ddir)
    frame_dirs = sorted(p for p in (ddir / "frames").iterdir() if p.is_dir())
    df = pd.concat(
        [frame_alignment(fd, tol_r, tol_v, close_pair_mode=close_pair) for fd in frame_dirs],
        ignore_index=True,
    )
    if save:
        out = ddir / "validation_report"
        out.mkdir(exist_ok=True)
        df.to_csv(out / "alignment_report.csv", index=False)
    return df


df = validate_dataset("clean_debug")
df

## Step 5 — Did anything fail?

Filter the DataFrame to non-passing rows. An empty result means every target aligned.

In [ ]:
df[~df["passed"]]

## Step 6 — Summary across all datasets

One row per dataset: target counts, isolated vs overlapping, pass count, the worst
peak-vs-label offset among isolated targets (should be ≤ 1–2 bins), and the weakest
peak-to-noise margin. Saved to `reports/validation/alignment_summary.csv`.

In [ ]:
def summarize(name):
    df = validate_dataset(name)
    t = df[df["target_id"] != -1]
    # Strictly validated = isolated AND visible (see frame_alignment).
    strict = t[(t["isolated"] == True) & (t["low_snr"] == False)]
    return {
        "dataset": name,
        "frames": df["frame"].nunique(),
        "targets": len(t),
        "strict": len(strict),
        "overlap": int((t["isolated"] == False).sum()),
        "low_snr": int((t["low_snr"] == True).sum()),
        "records_pass": int(df["passed"].sum()),
        "records_total": len(df),
        "max_abs_dr": int(strict["dr"].abs().max()) if len(strict) else 0,
        "max_abs_dv": int(strict["dv"].abs().max()) if len(strict) else 0,
        "median_cell_snr_db": float(t["cell_snr_db"].median()) if len(t) else np.nan,
        "all_pass": bool(df["passed"].all()),
    }


summary = pd.DataFrame([summarize(d) for d in DATASETS])
summary.to_csv(REPORTS_DIR / "alignment_summary.csv", index=False)
summary

## Step 7 — Three-panel visual for the report / slides

Clean, multi-target, and close-pair side by side. In the close-pair panel the two ✕ markers
sit almost on top of each other — that is the resolution limit (the weaker target is masked),
which is exactly what the `overlap` count in Step 6 measured. Saved as a PNG.

In [ ]:
def busiest_frame(name):
    """Return the frame dir with the most targets (so the panel isn't empty)."""
    best, best_n = None, -1
    for fd in sorted((DATA_ROOT / name / "frames").iterdir()):
        n = len(pd.read_csv(fd / "labels.csv"))
        if n > best_n:
            best, best_n = fd, n
    return best


fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, name in zip(axes, ["clean_debug", "multi_target_debug", "close_targets_debug"]):
    plot_frame(busiest_frame(name), ax=ax)
fig.tight_layout()
fig.savefig(REPORTS_DIR / "alignment_previews.png", dpi=150, bbox_inches="tight")
plt.show()
print("saved ->", REPORTS_DIR / "alignment_previews.png")

## Step 8 — Bonus: this also exposes the SNR problem

`snr_sweep_debug` cycles the *labelled* SNR from −10 to +20 dB. But look at the
`peak_to_floor_db` even for the −10 dB frames: the target peak is still tens of dB above the
noise floor. That is the FFT processing gain (~43 dB) — so the nominal SNR does **not** reflect
how detectable a target is, which is why the SNR definition needs recalibrating before it can
drive a meaningful Pd-vs-SNR curve.

In [ ]:
snr = validate_dataset("snr_sweep_debug")

def labelled_snr(frame):
    meta = json.loads((DATA_ROOT / "snr_sweep_debug" / "frames" / frame / "metadata.json").read_text())
    return meta["snr_db"]

snr = snr[snr["target_id"] != -1].copy()
snr["labelled_snr_db"] = snr["frame"].map(labelled_snr)
(snr[["frame", "labelled_snr_db", "peak_to_floor_db"]]
    .sort_values("labelled_snr_db")
    .reset_index(drop=True)
    .head(12))

## Conclusion

- Every **isolated** target across all scenarios peaks within ±1–2 bins of its label, the mask
  is set, and no frame has NaN/Inf values → **the labelling pipeline is spatially correct.**
- In `close_targets`, the weaker target of sub-resolution pairs is correctly flagged as
  unresolved (the radar's resolution limit), not as an error.
- The same run shows the nominal SNR overstates detectability (Step 8), motivating the SNR
  recalibration that follows.

**For the report:** *"To validate the synthetic labelling pipeline, the local maximum of the
RD magnitude map within a ±3-bin neighbourhood of each labelled cell was compared against the
stored bin indices and the segmentation mask. Across all scenarios, every isolated target
agreed within one resolution cell in both dimensions, with no invalid values, confirming that
the ground-truth labels are spatially faithful to the simulated signal."*